# 04 — Project 1: SMS Spam/Ham Classification

Runnable companion to **notes 16–17**. Three vectorization strategies on the same 5,572
messages, compared honestly.

| Section | |
|---|---|
| 1. Load the data | |
| 2. Clean + encode labels | |
| 3. Split **before** vectorizing | [note 16](../notes/16-Best-Practices-and-Data-Leakage.md) |
| 4. Approach A — Bag of Words + MultinomialNB | |
| 5. Approach B — TF-IDF + MultinomialNB (and why it loses) | |
| 6. Approach C — Average Word2Vec + RandomForest | |
| 7. The row-mismatch bug, found and fixed | |
| 8. Results and what they teach | |
| 9. The leakage experiment | |
| 10. `Pipeline` + `GridSearchCV` | |

## 0. Setup

In [ ]:
# !pip install pandas numpy scikit-learn nltk gensim tqdm matplotlib

import re, time
import numpy as np
import pandas as pd
import nltk

for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
print("ready")

## 1. Load the data

**SMS Spam Collection** — 5,572 messages, tab-separated, no header.
Download: <https://archive.ics.uci.edu/dataset/228/sms+spam+collection>

In [ ]:
# Adjust the path to wherever you saved the file
PATH = 'SMSSpamCollection'

messages = pd.read_csv(PATH, sep='\t', names=['label', 'message'])
print(messages.shape)
messages.head()

**Why `sep='\t'` and `names=[...]`?** The file is tab-separated (not comma) and has no
header row, so we supply the column names ourselves.

In [ ]:
print(messages['label'].value_counts())
print()
print(messages['label'].value_counts(normalize=True).round(4))

### ⚠️ ~13% spam — an imbalanced dataset

**Always predict "ham" and you score 86.6% accuracy while being useless.** Consequences:

- **Accuracy alone is not a valid metric.** Report precision/recall/F1 **on the spam class**.
- Use `stratify=y` when splitting.
- A **false positive** (real message sent to spam) is worse than a false negative here — you
  can delete spam, you cannot read a message you never saw. So optimise **precision**.

In [ ]:
baseline = messages['label'].value_counts(normalize=True).max()
print(f"majority-class baseline accuracy: {baseline:.4f}")
print("Any model must beat this to be worth anything.")

In [ ]:
for label in ['ham', 'spam']:
    print(f"--- {label.upper()} examples ---")
    for m in messages[messages['label'] == label]['message'].head(3):
        print("  ", m[:110])
    print()

## 2. Clean + encode labels

In [ ]:
ps   = PorterStemmer()
STOP = set(stopwords.words('english'))     # hoisted OUT of the loop -- see note 06

t0 = time.time()
corpus = []
for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])   # specials -> SPACE
    review = review.lower().split()                             # lowercase + tokenize
    review = [ps.stem(w) for w in review if w not in STOP]       # stopwords + stem
    corpus.append(' '.join(review))                              # back to a string

print(f"cleaned {len(corpus)} messages in {time.time()-t0:.2f}s")
for raw, clean in zip(messages['message'][:3], corpus[:3]):
    print("\nRAW  :", raw[:100])
    print("CLEAN:", clean[:100])

The stems look odd (`crazi`, `joke`, `entri`) — that is Porter working as designed
(note 04). The model does not read English; it needs every occurrence of `crazy` to become
the **same token**, and it does.

In [ ]:
y = pd.get_dummies(messages['label'], drop_first=True).values.ravel().astype(int)

print("y[:10] :", y[:10])
print("0 = ham, 1 = spam")
print("spam count:", y.sum(), "  ham count:", len(y) - y.sum())

`drop_first=True` keeps only the `spam` column — two categories need one column, and
keeping both gives you perfectly collinear features.

## 3. Split BEFORE vectorizing  ([note 16](../notes/16-Best-Practices-and-Data-Leakage.md))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    corpus, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print("train:", len(X_train), "  test:", len(X_test))
print("spam ratio  train: %.4f   test: %.4f" % (np.mean(y_train), np.mean(y_test)))
print("^ stratify=y kept the class balance identical in both halves")
print()
print("Note: we split the cleaned TEXT, not vectors. The vectorizer has not been fitted yet.")

## 4. Approach A — Bag of Words + Multinomial Naive Bayes

In [ ]:
cv = CountVectorizer(max_features=2500, ngram_range=(1, 2), binary=True)

X_train_bow = cv.fit_transform(X_train).toarray()   # fit_transform -> TRAIN (learns vocabulary)
X_test_bow  = cv.transform(X_test).toarray()        # transform ONLY -> TEST  (reuses it)

print("X_train_bow:", X_train_bow.shape)
print("X_test_bow :", X_test_bow.shape)
print("max value  :", X_train_bow.max(), "(binary=True, so never above 1)")
print()
print("some bigram features:",
      [f for f in cv.get_feature_names_out() if ' ' in f][:8])

In [ ]:
model_bow  = MultinomialNB().fit(X_train_bow, y_train)
y_pred_bow = model_bow.predict(X_test_bow)

acc_bow = accuracy_score(y_test, y_pred_bow)
print(f"accuracy: {acc_bow:.4f}   (baseline was {baseline:.4f})")
print()
print(classification_report(y_test, y_pred_bow, target_names=['ham', 'spam']))

In [ ]:
cm = confusion_matrix(y_test, y_pred_bow)
print(pd.DataFrame(cm,
                   index=['actual ham', 'actual spam'],
                   columns=['pred ham', 'pred spam']))

tn, fp, fn, tp = cm.ravel()
print(f"\nfalse positives (real message -> spam folder): {fp}   <- the expensive error")
print(f"false negatives (spam -> inbox)               : {fn}   <- merely annoying")

### Why Multinomial Naive Bayes for text?

- Its likelihood **is** a multinomial over word counts — the model matches the features.
- Handles high-dimensional sparse matrices without blinking.
- Extremely fast: one counting pass, no iteration.
- On small/medium text datasets it often beats logistic regression and SVM.

It requires **non-negative** features, which is why it works with BoW and TF-IDF but will
fail on Word2Vec (§6).

### 4.1 What did the model actually learn?

In [ ]:
names   = cv.get_feature_names_out()
log_odds = model_bow.feature_log_prob_[1] - model_bow.feature_log_prob_[0]   # spam vs ham

top_spam = np.argsort(log_odds)[-15:][::-1]
top_ham  = np.argsort(log_odds)[:15]

print("strongest SPAM indicators:", [names[i] for i in top_spam])
print()
print("strongest HAM  indicators:", [names[i] for i in top_ham])

If those lists look like real spam/ham vocabulary, the model is learning the right thing.
**Always inspect this** — if you see random names or artefacts, something is wrong.

## 5. Approach B — TF-IDF + Multinomial Naive Bayes

In [ ]:
tv = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))

X_train_tfidf = tv.fit_transform(X_train).toarray()
X_test_tfidf  = tv.transform(X_test).toarray()

model_tfidf  = MultinomialNB().fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f"accuracy: {acc_tfidf:.4f}")
print()
print(classification_report(y_test, y_pred_tfidf, target_names=['ham', 'spam']))

### ⚠️ TF-IDF scored **lower** than BoW. Why?

1. **Model/feature mismatch.** `MultinomialNB` assumes multinomial **counts**; TF-IDF's
   normalised fractional weights violate that assumption.
2. **Messages are very short.** TF-IDF's advantage is down-weighting words that saturate
   long documents — in a 12-word SMS, term frequency ≈ presence.
3. It became **more conservative**: near-perfect precision, much lower recall. A different
   operating point, not strictly worse.

**Match the vectorizer to the classifier.** Pair TF-IDF with Logistic Regression instead:

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

print(f"TF-IDF + LogisticRegression accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['ham', 'spam']))

## 6. Approach C — Average Word2Vec + Random Forest

Different preprocessing this time: **lemmatize, and KEEP the stopwords.** Word2Vec's whole
premise is that context words carry information, so stripping them removes the signal it
learns from.

In [ ]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tqdm.auto import tqdm

lemmatizer = WordNetLemmatizer()

corpus_w2v = []
for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i]).lower().split()
    review = [lemmatizer.lemmatize(w) for w in review]      # NO stopword removal
    corpus_w2v.append(' '.join(review))

words = [simple_preprocess(doc) for doc in corpus_w2v]      # LIST OF TOKEN LISTS
print("documents:", len(words))
print("first    :", words[0][:12])

## 7. 🐛 The row-mismatch bug

Before training, count the rows.

In [ ]:
empty = [i for i, doc in enumerate(words) if len(doc) == 0]

print("total documents :", len(words))
print("empty after clean:", len(empty))
print()
for i in empty:
    print(f"  row {i}: original = {messages['message'][i][:70]!r}")
    print(f"           cleaned  = {corpus_w2v[i]!r}")

**Messages made entirely of digits and punctuation.** `re.sub('[^a-zA-Z]', ' ', ...)`
strips every character, leaving an empty string, which produces **no word vectors**, so
`np.mean([], axis=0)` returns `NaN`.

If you drop those rows from `X` but not from `y`, every prediction afterwards is compared
against the wrong label — and nothing warns you.

### Three fixes, best first

In [ ]:
# (a) BEST: keep the digits -- a phone number IS spam signal
sample = messages['message'][empty[0]] if empty else "Free entry: call 87121 now"
print("with [^a-zA-Z]  :", repr(re.sub('[^a-zA-Z]',  ' ', sample).strip()))
print("with [^a-zA-Z0-9]:", repr(re.sub('[^a-zA-Z0-9]', ' ', sample).strip()))
print()
print("'Free entry, call 87121 now' -- the number is one of the strongest spam features")
print("in this dataset, and [^a-zA-Z] deletes it. Think before stripping digits.")

In [ ]:
# (b) guard the averaging function so an empty document yields zeros, not NaN
def avg_word2vec(doc, m):
    vecs = [m.wv[w] for w in doc if w in m.wv.index_to_key]
    if len(vecs) == 0:
        return np.zeros(m.vector_size)
    return np.mean(vecs, axis=0)          # axis=0: average DOWN the columns

# (c) or drop the rows -- from X AND y in the SAME operation
mask = np.array([len(d) > 0 for d in words])
print("mask keeps", mask.sum(), "of", len(mask), "rows")

### Train Word2Vec and build the feature matrix

In [ ]:
w2v = Word2Vec(words, vector_size=100, window=5, min_count=2,
               workers=4, sg=0, epochs=10, seed=RANDOM_STATE)

print("vocabulary :", len(w2v.wv.index_to_key))
print("documents  :", w2v.corpus_count)
print("vector dim :", w2v.wv['good'].shape if 'good' in w2v.wv else 'n/a')
print()
for w in ['free', 'call', 'love']:
    if w in w2v.wv:
        print(f"similar to {w!r}:", [x[0] for x in w2v.wv.most_similar(w, topn=5)])

In [ ]:
X_w2v = np.vstack([avg_word2vec(doc, w2v) for doc in tqdm(words, desc="averaging")])

print("X_w2v.shape:", X_w2v.shape)
print("y.shape    :", y.shape)
print("aligned?   ", X_w2v.shape[0] == y.shape[0], " <- guard (b) kept every row")
print()
print("NaNs remaining:", int(np.isnan(X_w2v).sum()))

**100 dense features instead of 2,500 sparse ones** — a 25× compression.

In [ ]:
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_w2v, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xw_train, yw_train)
y_pred_w2v = rf.predict(Xw_test)

acc_w2v = accuracy_score(yw_test, y_pred_w2v)
print(f"accuracy: {acc_w2v:.4f}")
print()
print(classification_report(yw_test, y_pred_w2v, target_names=['ham', 'spam']))

### Why Random Forest and not Naive Bayes here?

In [ ]:
try:
    MultinomialNB().fit(Xw_train, yw_train)
except ValueError as e:
    print("MultinomialNB failed:", e)

print()
print("Word2Vec vectors contain NEGATIVE numbers:", Xw_train.min().round(4))
print()
print("SPARSE counts (BoW, TF-IDF) -> MultinomialNB, LinearSVC, LogisticRegression")
print("DENSE embeddings (Word2Vec) -> RandomForest, LogisticRegression, SVM, GaussianNB")

## 8. Results

In [ ]:
results = pd.DataFrame({
    'Vectorizer': ['BoW (1,2) binary', 'TF-IDF (1,2)', 'TF-IDF (1,2)', 'Avg Word2Vec'],
    'Model':      ['MultinomialNB', 'MultinomialNB', 'LogisticRegression', 'RandomForest'],
    'Features':   [X_train_bow.shape[1], X_train_tfidf.shape[1],
                   X_train_tfidf.shape[1], X_w2v.shape[1]],
    'Type':       ['sparse', 'sparse', 'sparse', 'dense'],
    'Accuracy':   [acc_bow, acc_tfidf, accuracy_score(y_test, y_pred_lr), acc_w2v],
})
results['Accuracy'] = results['Accuracy'].round(4)
results.sort_values('Accuracy', ascending=False).reset_index(drop=True)

### What the table actually teaches

**1. The simplest method usually wins here.** BoW + Naive Bayes — the oldest technique in
this course — is at or near the top. Sophistication ≠ accuracy.

**2. Why?** Spam detection is nearly a **keyword problem**. `free`, `win`, `claim`, `txt`,
`prize` — their *presence* is the signal. You do not need to know what "free" *means*.

**3. Word2Vec's strengths do not pay off here.** Semantic similarity helps when meaning
matters. For keyword spotting, averaging 100 dimensions actively **dilutes** the "this exact
word is present" signal that BoW preserves.

**4. But look at the compression.** ~Equivalent accuracy from **100 dense features** instead
of 2,500 sparse ones. On a larger or more nuanced dataset (notebook 05), that pays off.

> **Always run the simple baseline first.** If BoW + Naive Bayes gets 98% in ten seconds,
> anything heavier has to justify itself against that number.

## 9. The leakage experiment

Note 16 says vectorizing before splitting inflates your score. Measure it.

In [ ]:
# WRONG: fit the vectorizer on the ENTIRE corpus, then split
cv_leak = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X_all   = cv_leak.fit_transform(corpus).toarray()       # <- sees test documents

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

acc_leaked = accuracy_score(yl_te, MultinomialNB().fit(Xl_tr, yl_tr).predict(Xl_te))

print(f"LEAKED (fit on everything) : {acc_leaked:.4f}")
print(f"CLEAN  (fit on train only) : {acc_tfidf:.4f}")
print(f"inflation                  : {acc_leaked - acc_tfidf:+.4f}")

On this large, easy dataset the gap is small — but **it is always in the optimistic
direction**, and the real cost is not the number. It is that every downstream decision
(which vectorizer, which n-gram range, which classifier) is then made on a corrupted signal.

The gap grows dramatically with smaller datasets, feature selection (`SelectKBest`), and
SMOTE applied before splitting.

## 10. `Pipeline` + `GridSearchCV` — leakage made structurally impossible

In [ ]:
pipe = Pipeline([
    ('vec', CountVectorizer()),
    ('clf', MultinomialNB()),
])

grid = GridSearchCV(
    pipe,
    {
        'vec__max_features': [1000, 2500, 5000],
        'vec__ngram_range':  [(1,1), (1,2)],
        'vec__binary':       [True, False],
        'clf__alpha':        [0.1, 0.5, 1.0],
    },
    cv=5, scoring='f1', n_jobs=-1, verbose=0,
)

t0 = time.time()
grid.fit(X_train, y_train)
print(f"searched {len(grid.cv_results_['params'])} combinations in {time.time()-t0:.1f}s")
print()
print("best params:", grid.best_params_)
print("best CV f1 :", round(grid.best_score_, 4))

Inside `GridSearchCV`, the `Pipeline` refits the vectorizer on **each fold's training
portion only** — automatically. Doing this by hand is where leakage sneaks back in.

In [ ]:
y_pred_best = grid.predict(X_test)      # the test set, touched exactly once

print(f"tuned test accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print()
print(classification_report(y_test, y_pred_best, target_names=['ham', 'spam']))

### Optimise for precision instead — because false positives cost the most

In [ ]:
grid_p = GridSearchCV(
    pipe,
    {'vec__max_features': [2500, 5000],
     'vec__ngram_range':  [(1,1), (1,2)],
     'clf__alpha':        [0.01, 0.1, 1.0]},
    cv=5, scoring='precision', n_jobs=-1,
)
grid_p.fit(X_train, y_train)

print("best params (precision):", grid_p.best_params_)
print()
print(classification_report(y_test, grid_p.predict(X_test), target_names=['ham','spam']))
print("Compare the spam-row precision with the f1-tuned model above.")

### Threshold tuning — an even more direct lever

In [ ]:
proba = grid.predict_proba(X_test)[:, 1]

print(f"{'threshold':>10} {'precision':>10} {'recall':>8} {'false pos':>10}")
for th in [0.5, 0.7, 0.9, 0.95, 0.99]:
    pred = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec  = tp / (tp + fn) if (tp + fn) else 0
    print(f"{th:>10.2f} {prec:>10.4f} {rec:>8.4f} {fp:>10}")

print("\nRaising the threshold trades recall for precision -- exactly the trade a")
print("spam filter wants. You do not need a new model to do this.")

---

## Exercises

1. Swap `PorterStemmer` for `WordNetLemmatizer` in §2 and re-run approach A. Better or worse?
2. Use `[^a-zA-Z0-9]` so phone numbers survive. Measure the accuracy change.
3. Use the **Google pre-trained** Word2Vec (300-dim) instead of training on 5,572 messages.
4. Add `class_weight='balanced'` to `LogisticRegression`. What moves — precision or recall?
5. Apply SMOTE **on the training split only** and compare. Then apply it before splitting and
   watch the leaked score jump.
6. Print 20 misclassified messages and read them. What do the errors have in common?

**Next notebook:** [`05-kindle-sentiment-project.ipynb`](05-kindle-sentiment-project.ipynb) —
a messier dataset, and a model that starts at 58%.